# Does the nearest-gauge advantage survive a SINGLE neighbour?

3 runs, ~2 h. Outputs go to `MyDrive/neural_hydro_runs/paper_completion/`. Idempotent.

**The one gap left in the paper's confound story.** The nearest-gauge input averages k=2 neighbours
while the true network averages 4.16, so averaging could reduce variance independently of distance.
Stratifying on the network's in-degree (`INDEGREE_CONFOUND.md`) shows the advantage at full size
where the network supplies <=2 parents — but the nearest-gauge input still averages **two** series
there, so a 2-over-1 variance advantage is not excluded.

**k=1 averages one series.** If the advantage survives, averaging is ruled out entirely.

A swept-distance arm cannot substitute: that control preserves in-degree, so its floor is ~76 km
against kNN2's 46.7 km. Distance and count are entangled there. k=1 is the only clean manipulation.

Note the arms are not nested — k=1 is *closer* (37.3 km vs 46.7) **and** *less averaged*, and those
push opposite ways. A value between 0 and +0.034 is genuinely ambiguous and will be reported as such.

Pre-registration: `experiments/topology_ablation/preregistration_knn1_count.md`.

**Runtime → Change runtime type → T4 GPU → Run all.**

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Config

In [ ]:
import os, glob
GITHUB_URL='https://github.com/Op-2005/neural_hydro.git'
DRIVE_CAMELS_PATH=''
OUT_SUBDIR='paper_completion'
AUTO=['/content/drive/MyDrive/datasets/camels_us','/content/drive/MyDrive/neural_hydro/datasets/camels_us',
      '/content/drive/MyDrive/neural_hydrology/datasets/camels_us','/content/drive/MyDrive/camels_us']
if not DRIVE_CAMELS_PATH:
    for cand in AUTO:
        if os.path.isdir(cand): DRIVE_CAMELS_PATH=cand; break
if not DRIVE_CAMELS_PATH or not os.path.isdir(DRIVE_CAMELS_PATH):
    raise RuntimeError(f'CAMELS not found. Tried: {AUTO}')
DRIVE_ROOT='/content/drive/MyDrive/neural_hydro_runs'
DRIVE_OUT=os.path.join(DRIVE_ROOT, OUT_SUBDIR)
os.makedirs(f'{DRIVE_OUT}/topology_ablation/component0', exist_ok=True)
SEEDS=[11,13,17]
print('CAMELS:',DRIVE_CAMELS_PATH); print('OUT   :',DRIVE_OUT)

## Cell 3 — Clone repo

In [ ]:
REPO_DIR='/content/nh'; import shutil
%cd /content
if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone --branch paper_writing {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -n 2

## Cell 4 — Install deps (pin numpy<2 / pandas 2.1.4)

In [ ]:
%cd {REPO_DIR}
!pip install -q -e . pynhd networkx 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps "numpy<2" "pandas==2.1.4" 2>&1 | tail -2
import sys
for m in list(sys.modules):
    if m.startswith('numpy') or m.startswith('pandas'): del sys.modules[m]
import numpy as np, pandas as pd, torch
print(f'numpy {np.__version__} pandas {pd.__version__} torch {torch.__version__} CUDA {torch.cuda.is_available()}')

## Cell 5 — Symlink data + runs, link in the baselines

In [ ]:
%cd {REPO_DIR}
import shutil, glob
RD=os.path.join(REPO_DIR,'datasets','camels_us'); os.makedirs(os.path.dirname(RD),exist_ok=True)
if os.path.islink(RD): os.unlink(RD)
elif os.path.isdir(RD): shutil.rmtree(RD,ignore_errors=True)
os.symlink(DRIVE_CAMELS_PATH, RD)
RR=os.path.join(REPO_DIR,'runs')
if os.path.islink(RR): os.unlink(RR)
elif os.path.isdir(RR): shutil.rmtree(RR,ignore_errors=True)
os.symlink(DRIVE_OUT, RR)
NEWD=f'{DRIVE_OUT}/topology_ablation/component0'; os.makedirs(NEWD,exist_ok=True)
SEARCH=[NEWD, f'{DRIVE_ROOT}/topology_ablation/component0']
for d in sorted(glob.glob(f'{DRIVE_ROOT}/*/topology_ablation/component0')):
    if d not in SEARCH: SEARCH.append(d)
def locate(nm):
    for root in SEARCH:
        p=f'{root}/{nm}'
        if os.path.isdir(p) and os.path.abspath(p)!=os.path.abspath(f'{NEWD}/{nm}'): return p
    return None
linked=0; miss=[]
for cond in ['L','L_upQ','L_upQknn2']:
    for s in SEEDS:
        nm=f'{cond}_component0_seed{s}'; dst=f'{NEWD}/{nm}'
        if os.path.exists(dst) or os.path.islink(dst): continue
        src=locate(nm)
        if src: os.symlink(src,dst); linked+=1
        else: miss.append(nm)
print('baselines linked:',linked)
if miss: print('MISSING (deltas will be unavailable):',miss)

## Cell 6 — GPU check

In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> T4 GPU.')
print('GPU:',torch.cuda.get_device_name(0))

## Cell 7 — Build the k=1 feature and train

Built through the same tested builder as every other feature (`--mode knn --knn-k 1`), which
excludes true parents and emits all 183 basins with a `'date'`-named index. Fails loudly rather
than training into a missing file.

In [ ]:
%cd {REPO_DIR}
import pickle, subprocess, time
FEAT='experiments/topology_ablation/features'
B=f'{REPO_DIR}/runs/topology_ablation/component0'
N_BASINS=183
def named_ok(p,n=N_BASINS):
    if not os.path.isfile(p): return False
    d=pickle.load(open(p,'rb'))
    if d[next(iter(d))].index.name!='date': return False
    if len(d)<n: print(f'  [rebuild] {os.path.basename(p)} covers {len(d)}/{n}'); return False
    return True
def done(cond,s): return os.path.isfile(f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv')

fp=f'{FEAT}/upstream_q_knn1_component0_lag1.p'
if not named_ok(fp):
    !python experiments/topology_ablation/build_distance_control.py --network component0 --mode knn --knn-k 1 --lag-days 1
if not named_ok(fp): raise RuntimeError('k=1 feature did not build; refusing to train')
print('k=1 feature ok')

for s in SEEDS:
    if done('L_upQknn1',s): print(f'  [skip] seed {s}'); continue
    print(f'  [run ] L_upQknn1 seed {s} ...',flush=True); t0=time.time()
    r=subprocess.run(['python','experiments/topology_ablation/run_upstream_feature.py',
        '--network','component0','--seed',str(s),'--device','cuda:0','--epochs','30',
        '--feature-file',fp,'--cond-name','L_upQknn1'],capture_output=True,text=True)
    ok=done('L_upQknn1',s); print(f'  [{"ok" if ok else "FAIL"}] seed {s} ({time.time()-t0:.0f}s)')
    if not ok: print(r.stdout[-1200:]); print(r.stderr[-1200:])

## Cell 8 — Verdict

In [ ]:
%cd {REPO_DIR}
import pandas as pd, numpy as np, pickle
from scipy.stats import wilcoxon
def nse(c_,s):
    p=f'{B}/{c_}_component0_seed{s}/test/model_epoch030/test_metrics.csv'
    return pd.read_csv(p,dtype={'basin':str}).set_index('basin')['NSE'] if os.path.isfile(p) else None
fe=pickle.load(open(f'{FEAT}/upstream_q_component0_lag1.p','rb'))
CONN=sorted([b for b,v in fe.items() if float(np.nanmax(np.abs(v.values)))>0])
def vs_network(cond):
    meds,ps=[],[]
    for s in SEEDS:
        A,G=nse(cond,s),nse('L_upQ',s)
        if A is None or G is None: return None,None
        d=np.array([A[b]-G[b] for b in CONN])
        meds.append(np.median(d)); ps.append(wilcoxon(d,alternative='greater')[1])
    return meds,max(ps)
print('Paired advantage over the TRUE NETWORK, connected basins (n=150), per seed:')
for cond,lab in [('L_upQknn2','kNN k=2 (2 neighbours, 46.7 km)'),('L_upQknn1','kNN k=1 (ONE neighbour, 37.3 km)')]:
    m,wp=vs_network(cond)
    if m is None: print(f'  {lab:36s} MISSING'); continue
    print(f'  {lab:36s} {[f"{x:+.4f}" for x in m]}  mean {np.mean(m):+.4f}  weakest-seed p={wp:.4f}')
m1,p1=vs_network('L_upQknn1'); m2,_=vs_network('L_upQknn2')
if m1:
    a1,a2=np.mean(m1),np.mean(m2)
    print('\n=== PRE-REGISTERED VERDICT ===')
    if a1>0 and p1<0.05:
        print(f'  COUNT IS NOT THE MECHANISM. A single neighbour still beats the network')
        print(f'  ({a1:+.4f}, weakest-seed p={p1:.4f}). Averaging is ruled out entirely.')
        if a1 < a2-0.005:
            print(f'  Note: k=1 ({a1:+.4f}) sits below k=2 ({a2:+.4f}), so averaging may contribute')
            print(f'  even though it is not required. Report both; the arms are not nested.')
    elif a1<=0:
        print('  FALSIFIES THE DISTANCE READING. The closest possible neighbour does NOT beat the')
        print('  network, so something other than proximity is operating. Reopen the mechanism.')
    else:
        print(f'  AMBIGUOUS: positive ({a1:+.4f}) but weakest-seed p={p1:.4f}. Underpowered; report as such.')

## Cell 9 — Persistence check

In [ ]:
ok=miss=0
for s in SEEDS:
    p=f'{DRIVE_OUT}/topology_ablation/component0/L_upQknn1_component0_seed{s}/test/model_epoch030/test_metrics.csv'
    g=os.path.isfile(p); ok+=g; miss+=(not g)
    print(f'  {"OK  " if g else "MISS"} L_upQknn1 seed {s}')
print(f'\n{ok} present, {miss} missing (expect 3)')

## Done

Paste **Cell 8** and **Cell 9** back.

This closes the last open alternative explanation for the paper's headline. Either averaging is
ruled out entirely, or the paper reports how much of the advantage it accounts for.